# 01 -- Validation and diagnostics

Neutrino-induced muon background for EarthShine.  This notebook checks every
ingredient separately against the literature before anything is combined, and
produces the diagnostic figures.

**Contents**
1. Run the unit tests
2. Neutrino flux: analytic model vs. the digitised IceCube unfolding
3. Cross sections
4. Inelasticity
5. Muon energy loss and range
6. Muon yield from 1 m^3 of rock
7. The key internal consistency check
8. Literature comparison: up-going muon flux
9. Where does the background come from?

**References**
* IceCube atmospheric nu_mu unfolding: <https://icecube.wisc.edu/news/research/2014/09/an-improved-measurement-of-atmospheric-neutrino-flux-in-icecube/>, arXiv:[1409.4535](https://arxiv.org/abs/1409.4535)
* Chirkin CORSIKA fit: arXiv:[hep-ph/0407078](https://arxiv.org/abs/hep-ph/0407078)
* Formaggio & Zeller cross-section review: arXiv:[1305.7513](https://arxiv.org/abs/1305.7513)
* IceCube cross-section measurement: arXiv:[1711.08119](https://arxiv.org/abs/1711.08119)
* CSMS NLO QCD cross sections: arXiv:[1106.3723](https://arxiv.org/abs/1106.3723)
* PDG neutrino cross-section review: <https://pdg.lbl.gov/2025/reviews/rpp2025-rev-nu-cross-sections.pdf>
* Feng, Smolinsky & Tanedo (the EarthShine signal paper): arXiv:[1509.07525](https://arxiv.org/abs/1509.07525)

In [ ]:
# --- make `nubkg` importable no matter where this notebook is opened from ----
# Looks for the nubkg package next to the notebook, then one level up, then
# falls back to whatever is already installed on sys.path.
import sys, pathlib

def _bootstrap():
    here = pathlib.Path.cwd()
    for cand in [here, *here.parents][:4]:
        if (cand / "nubkg" / "__init__.py").exists():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    return None

_root = _bootstrap()
print("package root:", _root or "not found next to the notebook -- "
      "relying on an installed nubkg")

import numpy as np
import matplotlib.pyplot as plt
import nubkg as nb
from nubkg import plots as P

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.titlesize": 10, "figure.autolayout": True})
E_NU_RANGE = (10.0, 1.0e6)     # the range we care about

## 1. Unit tests

Run these first; nothing below is meaningful if they fail.

In [ ]:
print(np.__version__)

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pytest", "--pyargs",
                      "nubkg.test_nu_background", "-q", "--no-header"],
                     cwd=_root or ".", capture_output=True, text=True).stdout[-2000:])

## 2. Neutrino flux

Two independent routes:

* `TabulatedFlux` -- your digitised IceCube unfolding.  A *measurement*.
* `ChirkinAtmospheric` -- an analytic fit to CORSIKA air-shower simulations.

**These are not circular.** CORSIKA takes as input the primary cosmic-ray
spectrum and composition plus a hadronic interaction model; it never sees
neutrino-telescope data.  So the agreement below is a real cross-check.

The shipped CSV is a placeholder read off the figure by eye -- swap in your own.

In [ ]:
tab = nb.TabulatedFlux.from_csv(
    nb.data_path("ic59_atmospheric_numu_APPROX.csv"),
    label="IC-59 unfolding (PLACEHOLDER digitisation)", flux_unit="E2Phi")
chirkin = nb.ChirkinAtmospheric()
astro = nb.AstrophysicalPowerLaw()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
P.plot_flux([chirkin, astro], cos_zenith=-0.5, ax=axes[0])
P.plot_flux_e2([chirkin, astro], cos_zenith=-0.5, ax=axes[1], tabulated=tab)
axes[1].set_ylim(1e-10, 1e0)
plt.show()

The comparison to make quantitative: the tabulated points are averaged over
the up-going sky, so compare them against the model's *hemisphere average*,
not against a single zenith.

In [ ]:
print(f"{'E_nu [GeV]':>12} {'table':>12} {'Chirkin<up>':>12} {'ratio':>8}")
for e in (3e2, 1e3, 1e4, 1e5, 5e5):
    t = sum(tab(e, -0.5, s) for s in ("nu", "nubar"))
    m = sum(chirkin.zenith_averaged(e, s, "up")[0] for s in ("nu", "nubar"))
    print(f"{e:12.1e} {t:12.3e} {m:12.3e} {t/m:8.2f}")

### Zenith dependence

The tabulated flux has none (it is a sky average), so we graft the model's
shape onto the measured normalisation with `zenith_shape_from`.  The
enhancement toward the horizon is a factor of several above the pion critical
energy and is not negligible, because the cylinder also presents its largest
projected area near the horizon.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
P.plot_zenith_dependence(chirkin, ax=axes[0])

hybrid = nb.TabulatedFlux.from_csv(
    nb.data_path("ic59_atmospheric_numu_APPROX.csv"),
    label="IC-59 norm x Chirkin zenith shape", flux_unit="E2Phi",
    zenith_shape_from=chirkin, shape_hemisphere="up")
cz = np.linspace(-1, -0.01, 100)
for e in (1e3, 1e4):
    y = sum(hybrid(np.full_like(cz, e), cz, s) for s in ("nu", "nubar"))
    axes[1].plot(cz, y, label=f"hybrid, {e:g} GeV")
    y2 = sum(chirkin(np.full_like(cz, e), cz, s) for s in ("nu", "nubar"))
    axes[1].plot(cz, y2, "--", label=f"Chirkin, {e:g} GeV")
axes[1].set_yscale("log"); axes[1].set_xlabel(r"$\cos\theta$")
axes[1].set_ylabel(r"$d\Phi/dE$"); axes[1].legend(fontsize=7); axes[1].grid(alpha=.25)
plt.show()

## 3. Cross sections

Table-driven, so you can drop in your digitisation of Formaggio & Zeller Fig 9
(low energy) and the IceCube Nature paper Fig 1 (high energy):

```python
sigma = nb.CrossSection.from_csv("mypoints.csv", sigma_unit="cm2_per_GeV")
```

The built-in default is anchored on the PDG world average below 350 GeV and on
the NLO QCD value at 10 TeV, with a log-log bridge between.  The bridge region
carries the largest model dependence and is shaded accordingly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
P.plot_cross_sections(ax=axes[0])
P.plot_cross_sections(ax=axes[1], per_energy=True)
axes[1].set_ylim(1e-39, 2e-38)
axes[1].annotate("PDG world averages\n0.677 / 0.334 e-38", xy=(3e2, 7e-39),
                 fontsize=7)
plt.show()

s = nb.default_cc("nu")
for e in (10, 100, 350, 1e3, 1e4, 1e5, 1e6):
    print(f"E = {e:8.0e} GeV   sigma_CC = {s(e):.3e} cm^2   "
          f"sigma/E = {s(e)/e:.3e}   (+-{100*nb.sigma_uncertainty(e):.0f}%)")

## 4. Inelasticity

Only the CC channel makes a muon, and the muon takes only $u = 1-y$ of the
neutrino energy.  The quark-parton model gives a flat $d\sigma/dy$ for $\nu$
($\langle y\rangle = 0.5$) and $3(1-y)^2$ for $\bar\nu$
($\langle y\rangle = 0.25$) -- the same valence assumption that produces the
$\sigma_{\bar\nu}/\sigma_\nu \to 1/3$ low-energy limit, so it is internally
consistent with the cross sections above.

Both quantities the calculation needs are analytic, which is why no Monte Carlo
over $y$ is required anywhere.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
P.plot_inelasticity(nb.Inelasticity("qpm"), ax=axes[0])
z = np.linspace(0, 1, 300)
for sp in ("nu", "nubar"):
    axes[1].plot(z, nb.Inelasticity("qpm").survival(z, sp), label=sp)
axes[1].set_xlabel("z"); axes[1].set_ylabel("S(z) = P(u > z)")
axes[1].set_title("survival function used by the arriving spectrum", fontsize=10)
axes[1].legend(fontsize=8); axes[1].grid(alpha=.25)
plt.show()

## 5. Muon energy loss and range

$-dE/dX = a + bE$, integrated in closed form.  The critical energy $a/b$ is
~500 GeV; below it the range is essentially $(E-E_{\rm th})/a$ and grows
linearly, above it radiative losses take over and the range grows only
logarithmically.  **This is what sets the size of the contributing rock volume**
-- metres at 10 GeV, hundreds of metres to a kilometre at TeV.

In [ ]:
from nubkg.muon_transport import ROCK_LOSS, WATER_LOSS
fig, ax = plt.subplots(figsize=(6, 4))
P.plot_muon_range([ROCK_LOSS], e_thr=(1., 10., 100., 1000.), ax=ax)
plt.show()

print("Check against 1509.07525 Eq. (32): a 1 TeV muon with E_th = 50 GeV")
print(f"  travels {WATER_LOSS.range_cm(1e3, 50., 1.0)/1e5:.2f} km in water "
      "(paper says 2.5 km)")
print(f"  travels {ROCK_LOSS.range_cm(1e3, 50., 2.65)/1e5:.2f} km in standard rock")
print()
for e_thr in (10., 100., 1000.):
    for e0 in (3*e_thr, 10*e_thr):
        print(f"  E_prod={e0:8.0f}  E_thr={e_thr:6.0f}  ->  "
              f"{ROCK_LOSS.range_cm(e0, e_thr, 2.65):9.1f} cm = "
              f"{ROCK_LOSS.range_cm(e0, e_thr, 2.65)/100:7.1f} m of rock")

The energy-loss bracket is carried as an explicit systematic, because a
*constant* $b$ is a compromise -- the true $b$ rises with energy.

In [ ]:
soft, nom, hard = ROCK_LOSS.bracket()
for l in (soft, nom, hard):
    print(f"{l.label:45s} b={l.b:.2e}  R(1 TeV -> 10 GeV) = "
          f"{l.range_cm(1e3, 10., 2.65)/100:6.1f} m")

## 6. Muon yield from a cubic metre of rock

Production only -- says nothing about whether the muon reaches anything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax, d = P.plot_rock_yield(ax=axes[0], e_nu_range=E_NU_RANGE, hemisphere="up")
ax, d2 = P.plot_rock_yield(ax=axes[1], e_nu_range=E_NU_RANGE, hemisphere="all")
axes[1].set_title("all directions")
plt.show()

tot = np.trapezoid(d["dN_dEmu_per_m3_per_s"], d["e_mu"]) * nb.SEC_PER_YEAR
print(f"muons produced per m^3 per year above 10 GeV (up-going hemisphere): {tot:.3e}")
print(f"-> one such muon per {1/tot:.3g} m^3-years of rock")

## 7. The key internal consistency check

The arriving spectrum is derived by turning the integral over path length into
an integral over arriving energy:

$$\frac{d\Phi_\mu}{dE_\mu} = \frac{N_A}{a + bE_\mu}\int dE_\nu\,
\phi_\nu\,\sigma_{\rm CC}\,\big[S(E_\mu/E_\nu) - S(E_{\rm cap}/E_\nu)\big]$$

Integrating this from $E_{\rm thr}$ upward must reproduce the familiar
$n_N \sigma \langle R_\mu\rangle$ form.  The two are coded independently, so
agreement is a real check on the derivation.

In [ ]:
for e_thr in (10., 100., 1000.):
    cz = np.array([-0.9, -0.5, -0.1])
    e_mu = np.geomspace(e_thr, 1e6, 900)
    spec = nb.arriving_muon_spectrum(e_mu, cz, e_nu_range=E_NU_RANGE, n_energy=800)
    a = np.trapezoid(spec, e_mu, axis=1)
    b = nb.integrated_muon_flux(e_thr, cz, e_nu_range=E_NU_RANGE, n_energy=800)
    print(f"E_thr = {e_thr:7.0f} GeV   max |spectrum/range - 1| = "
          f"{np.max(np.abs(a/b - 1)):.2e}")

## 8. Literature comparison: the up-going muon flux

The single most useful external check.  Super-Kamiokande and MACRO both measure
the up-going through-going muon flux above ~1.6 GeV at
$\sim 1.5-2\times10^{-13}$ cm$^{-2}$s$^{-1}$sr$^{-1}$, and that number
convolves flux $\times$ cross section $\times$ range into one measurement.

Note the measured value *includes* $\nu_\mu\to\nu_\tau$ oscillation, a ~20%
suppression for through-going muons, which this calculation does not apply --
so we should land on the low side of the measurement, not above it.

In [ ]:
cz = np.linspace(-1.0, -0.02, 60)
phi = nb.integrated_muon_flux(1.6, cz, e_nu_range=(1.0, 1e6), n_energy=600)
mean = np.trapezoid(phi, cz) / (cz[-1] - cz[0])
print(f"hemisphere-averaged Phi_mu(>1.6 GeV) = {mean:.2e} cm^-2 s^-1 sr^-1")
print("measured (Super-K, MACRO)             ~ 1.5-2e-13")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cz, phi, lw=1.6)
ax.axhspan(1.5e-13, 2.0e-13, alpha=.2, color="green",
           label="Super-K / MACRO band")
ax.set_xlabel(r"$\cos\theta_{\rm zenith}$")
ax.set_ylabel(r"$\Phi_\mu(>1.6\,$GeV$)$")
ax.set_yscale("log"); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.show()

## 9. Where does the background come from?

This decides which parts of the flux and cross-section tables actually matter,
and therefore where digitisation effort is worth spending.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, e_thr in zip(axes, (10., 100., 1000.)):
    _, info = P.plot_integrand_decomposition(e_thr, -0.5, ax=ax,
                                             e_nu_range=E_NU_RANGE)
    print(f"E_thr = {e_thr:6.0f} GeV:  median E_nu = {info['median_E_nu']:8.0f} GeV, "
          f"90% below {info['e_nu_90pct']:9.0f} GeV")
plt.show()

**Read this carefully before trusting the absolute number.** If the median
$E_\nu$ sits below ~600 GeV, the Chirkin parameterisation is extrapolating
(it was fitted over 600 GeV - 60 TeV) and the digitised IceCube points are
below their own lowest energy too. That region needs either Frejus data,
Honda tables, or MCEq.